# Scraping Harga Emas

## Library

In [109]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time
import polars as pl
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Tes pakai API logam mulia

In [36]:
# History dengan filter gabungan
response = requests.get("https://logam-mulia-api.iamutaki.workers.dev/api/prices/anekalogam/history?page=2&length=100&materialType=ANTAM&weight=1")
print(response.json())

{'success': True, 'data': [], 'pagination': {'page': 2, 'length': 100, 'total': 0, 'totalPages': 0}}


In [83]:
response = requests.get("https://logam-mulia-api.iamutaki.workers.dev/api/prices/hargaemas-org/history?page=5")
print(response.json())  # kalau responsenya JSON

data = response.json()  # langsung jadi dict/list Python

# # Dari file .json
# with open("data.json", "r") as f:
#     data = json.load(f)

# Lihat strukturnya dulu — ini WAJIB dilakukan
print(type(data))        # dict atau list?
print(data.keys())       # kalau dict, kunci-kuncinya apa?
print(data['data']) 
df = pd.DataFrame(data['data'])
display(df)

{'success': True, 'data': [], 'pagination': {'page': 5, 'length': 20, 'total': 20, 'totalPages': 1}}
<class 'dict'>
dict_keys(['success', 'data', 'pagination'])
[]


""


In [79]:
print(df['materialType'].unique())

<StringArray>
[       'SENTRA BUYBACK',          'BATIK SERIES', 'BABY SERIES INVESTASI',
  'BABY SERIES TUMBUHAN',       'UBS HELLO KITTY',      'LOTUS ARCHI GIFT',
   'UBS MICKEY FULLBODY',              'UBS ANNA',              'UBS ELSA',
            'UBS DISNEY',           'LOTUS ARCHI',   'ANTAM NON PEGADAIAN',
     'ANTAM MULIA RETRO',                   'UBS',                 'ANTAM',
       'BABY GALERI G24',             'DINAR G24',             'GALERI 24']
Length: 18, dtype: str


In [ ]:
import requests
import pandas as pd
import os
import time

# 1. Tentukan folder penyimpanan
folder_name = 'data_emas_history'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"Folder '{folder_name}' berhasil dibuat.")

# 2. Daftar semua source berdasarkan gambar yang Anda berikan
# Beberapa source mungkin tidak memiliki history (seperti sakumas, treasury, dll)
# namun kita masukkan semua untuk pengecekan otomatis.
sources = [
    "anekalogam", "hargaemas-org", "lakuemas", "sakumas", "kursdolar", 
    "cermati", "indogold", "hargaemas-net", "hargaemas-com", "treasury", 
    "logammulia", "emasku", "hartadinataabadi", "galeri24", "sampoernagold", 
    "bankbsi", "brankaslm", "pegadaian"
]

base_url = "https://logam-mulia-api.iamutaki.workers.dev/api/prices"

print("Memulai proses pengambilan data...")

# 3. Looping untuk mengambil data dari setiap endpoint
for source in sources:
    history_url = f"{base_url}/{source}/history"
    
    try:
        print(f"Mengambil data untuk: {source}...", end=" ")
        response = requests.get(history_url, timeout=10)
        
        # Cek apakah request berhasil
        if response.status_code == 200:
            data_json = response.json()
            
            # Pastikan kunci 'data' ada dan tidak kosong
            if 'data' in data_json and data_json['data']:
                df = pd.DataFrame(data_json['data'])
                
                # Simpan ke CSV
                file_path = os.path.join(folder_name, f"history_{source}.csv")
                df.to_csv(file_path, index=False)
                print(f"BERHASIL (Tersimpan di {file_path})")
            else:
                print("GAGAL (Data kosong/tidak tersedia)")
        else:
            print(f"GAGAL (Status Code: {response.status_code})")
            
    except Exception as e:
        print(f"ERROR: {str(e)}")
    
    # Beri jeda sedikit agar tidak membebani server (opsional)
    time.sleep(1)

print("\nProses selesai.")

Folder 'data_emas_history' berhasil dibuat.
Memulai proses pengambilan data...
Mengambil data untuk: anekalogam... BERHASIL (Tersimpan di data_emas_history\history_anekalogam.csv)
Mengambil data untuk: hargaemas-org... BERHASIL (Tersimpan di data_emas_history\history_hargaemas-org.csv)
Mengambil data untuk: lakuemas... BERHASIL (Tersimpan di data_emas_history\history_lakuemas.csv)
Mengambil data untuk: sakumas... GAGAL (Status Code: 404)
Mengambil data untuk: kursdolar... BERHASIL (Tersimpan di data_emas_history\history_kursdolar.csv)
Mengambil data untuk: cermati... BERHASIL (Tersimpan di data_emas_history\history_cermati.csv)
Mengambil data untuk: indogold... BERHASIL (Tersimpan di data_emas_history\history_indogold.csv)
Mengambil data untuk: hargaemas-net... BERHASIL (Tersimpan di data_emas_history\history_hargaemas-net.csv)
Mengambil data untuk: hargaemas-com... BERHASIL (Tersimpan di data_emas_history\history_hargaemas-com.csv)
Mengambil data untuk: treasury... GAGAL (Status Code:

In [80]:
import requests

sources = [
    "anekalogam", "hargaemas-org", "lakuemas", "sakumas", "kursdolar", 
    "cermati", "indogold", "hargaemas-net", "hargaemas-com", "treasury", 
    "logammulia", "emasku", "hartadinataabadi", "galeri24", "sampoernagold", 
    "bankbsi", "brankaslm", "pegadaian"
]

base_url = "https://logam-mulia-api.iamutaki.workers.dev/api/prices"

print("--- Pengecekan Pagination (totalPages > 1) ---")

for source in sources:
    history_url = f"{base_url}/{source}/history"
    
    try:
        # Kita set length=20 (default) supaya dapet info pagination yang normal
        response = requests.get(history_url, timeout=10)
        
        if response.status_code == 200:
            res_json = response.json()
            pagination = res_json.get('pagination', {})
            
            # Ambil nilai totalPages secara eksplisit
            # total_items = pagination.get('total', 0) # Ini yang 86 di galeri24
            real_total_pages = pagination.get('totalPages', 0) # Ini yang harusnya 5
            
            if real_total_pages > 1:
                print(f"{source}: iya (Halaman: {real_total_pages})")
            else:
                print(f"{source}: tidak")
        else:
            print(f"{source}: tidak (error {response.status_code})")
            
    except Exception:
        print(f"{source}: tidak (request error)")

print("---------------------------------------------")

--- Pengecekan Pagination (totalPages > 1) ---
anekalogam: tidak
hargaemas-org: tidak
lakuemas: tidak
sakumas: tidak (error 404)
kursdolar: tidak
cermati: iya (Halaman: 2)
indogold: tidak
hargaemas-net: tidak
hargaemas-com: iya (Halaman: 2)
treasury: tidak (error 404)
logammulia: iya (Halaman: 2)
emasku: tidak
hartadinataabadi: tidak
galeri24: iya (Halaman: 5)
sampoernagold: tidak (error 404)
bankbsi: tidak
brankaslm: tidak
pegadaian: tidak (error 404)
---------------------------------------------


## Langsung dari hargaemas.org

In [103]:
def scrape_harga_emas(tahun, bulan, tanggal):
    """
    Scrape harga emas 1 gram dari harga-emas.org.
    
    Parameters:
        tahun  (int): contoh 2026
        bulan  (int): 1-12
        tanggal (int): 1-31
    
    Returns:
        dict berisi tanggal dan harga Antam, Pegadaian, Pluang (1 gram)
        None jika halaman tidak tersedia / data tidak ditemukan
    """
    nama_bulan = BULAN_ID[bulan]
    url = f"https://harga-emas.org/history-harga/{tahun}/{nama_bulan}/{tanggal:02d}"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] HTTP {resp.status_code} - skip")
            return None
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        
        # Cari semua baris tabel
        rows = soup.select('table tbody tr')
        
        if not rows:
            print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] Tabel tidak ditemukan")
            return None
        
        # Baris ke-10 (index 9) = 1 gram
        # Kolom: [0] satuan, [1] Antam, [2] Pegadaian, [3] Pluang
        row_1gram = rows[9]
        cols = row_1gram.find_all('td')
        
        def parse_harga(text):
            """Ubah 'Rp2.796.000' → 2796000 (integer)"""
            return int(text.replace('Rp', '').replace('.', '').replace(',', '').strip())
        
        return {
            'tanggal'   : f"{tahun}-{bulan:02d}-{tanggal:02d}",
            'antam'     : parse_harga(cols[1].text),
            'pegadaian' : parse_harga(cols[2].text),
            'pluang'    : parse_harga(cols[3].text),
        }
    
    except Exception as e:
        print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] Error: {e}")
        return None
    
# ─── Contoh 2: Scrape range tanggal (misal 1 bulan penuh) ───────────────────
def scrape_range(start_date: str, end_date: str, delay: float = 1.0):
    """
    Scrape harga emas untuk rentang tanggal.
    
    Parameters:
        start_date (str): format 'YYYY-MM-DD'
        end_date   (str): format 'YYYY-MM-DD'
        delay      (float): jeda antar request dalam detik (default 1s, jangan < 0.5)
    
    Returns:
        DataFrame pandas
    """
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end   = datetime.strptime(end_date,   '%Y-%m-%d')
    
    results = []
    current = start
    
    while current <= end:
        print(f"Fetching {current.strftime('%Y-%m-%d')}...", end=' ')
        row = scrape_harga_emas(current.year, current.month, current.day)
        if row:
            results.append(row)
            print(f"✓ Antam: Rp{row['antam']:,}")
        else:
            print("(tidak ada data)")
        
        current += timedelta(days=1)
        time.sleep(delay)  # jeda agar tidak membebani server
    
    df = pd.DataFrame(results)
    if not df.empty:
        df['tanggal'] = pd.to_datetime(df['tanggal'])
        df = df.sort_values('tanggal').reset_index(drop=True)
    
    return df

BULAN_ID = {
    1: 'Januari', 2: 'Februari', 3: 'Maret', 4: 'April',
    5: 'Mei', 6: 'Juni', 7: 'Juli', 8: 'Agustus',
    9: 'September', 10: 'Oktober', 11: 'November', 12: 'Desember'
}


# Ambil data April 2026
df = scrape_range('2026-02-01', '2026-05-03')
print(f"\nTotal data: {len(df)} hari")
df.head(10)
df.to_csv("data_emas.csv", index=False)

Fetching 2026-02-01...   [2026/Februari/01] HTTP 404 - skip
(tidak ada data)
Fetching 2026-02-02...   [2026/Februari/02] HTTP 404 - skip
(tidak ada data)


KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://harga-emas.org/history-harga/2026/Mei/02"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

resp = requests.get(url, headers=headers, timeout=10)
print(f"Status: {resp.status_code}")

soup = BeautifulSoup(resp.text, 'html.parser')

# Cek semua tabel yang ada
tables = soup.find_all('table')
print(f"\nJumlah tabel ditemukan: {len(tables)}")

for i, table in enumerate(tables):
    rows = table.find_all('tr')
    print(f"\n--- Tabel {i} ({len(rows)} rows) ---")
    for j, row in enumerate(rows[:20]):
        cols = [td.get_text(strip=True) for td in row.find_all(['td','th'])]
        print(f"  Row {j}: {cols}")

Status: 307

Jumlah tabel ditemukan: 2

--- Tabel 0 (8 rows) ---
  Row 0: ['Prev', 'Mei2026', 'Next']
  Row 1: ['Ming', 'Sen', 'Sel', 'Rab', 'Kam', 'Jum', 'Sab']
  Row 2: ['26', '27', '28', '29', '30', '1', '2']
  Row 3: ['3', '4', '5', '6', '7', '8', '9']
  Row 4: ['10', '11', '12', '13', '14', '15', '16']
  Row 5: ['17', '18', '19', '20', '21', '22', '23']
  Row 6: ['24', '25', '26', '27', '28', '29', '30']
  Row 7: ['31', '1', '2', '3', '4', '5', '6']

--- Tabel 1 (13 rows) ---
  Row 0: ['Satuan', 'Harga Emas Logam Mulia Antam', 'Harga Emas Logam Mulia Pegadaian', 'Harga Emas Pluang']
  Row 1: ['Gram', 'per Gram (Rp)', 'per Gram (Rp)', 'per Batangan (Rp)']
  Row 2: ['1000', 'Rp2.736.600.000', 'Rp2.702.112.000', 'Rp2.644.818.000']
  Row 3: ['500', 'Rp1.368.320.000', 'Rp1.351.056.000', 'Rp1.322.409.000']
  Row 4: ['250', 'Rp684.265.000', 'Rp675.529.000', 'Rp661.204.500']
  Row 5: ['100', 'Rp273.812.000', 'Rp270.877.000', 'Rp264.481.800']
  Row 6: ['50', 'Rp136.945.000', 'Rp135.505.000

In [94]:
def scrape_harga_emas(tahun, bulan, tanggal):
    BULAN_ID = {
        1: 'Januari', 2: 'Februari', 3: 'Maret', 4: 'April',
        5: 'Mei', 6: 'Juni', 7: 'Juli', 8: 'Agustus',
        9: 'September', 10: 'Oktober', 11: 'November', 12: 'Desember'
    }
    
    nama_bulan = BULAN_ID[bulan]
    url = f"https://harga-emas.org/history-harga/{tahun}/{nama_bulan}/{tanggal:02d}"
    
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    # allow_redirects=True agar follow redirect 307
    resp = requests.get(url, headers=headers, timeout=10, allow_redirects=True)
    print(f"  Final URL: {resp.url} | Status: {resp.status_code}")
    
    if resp.status_code != 200:
        return None
    
    soup = BeautifulSoup(resp.text, 'html.parser')
    tables = soup.find_all('table')
    
    if len(tables) < 2:
        print("  Tabel data tidak ditemukan")
        return None
    
    # Tabel index 1 = tabel harga
    rows = tables[1].find_all('tr')
    
    hasil = []
    for row in rows[2:]:  # skip 2 baris header
        cols = [td.get_text(strip=True) for td in row.find_all('td')]
        if len(cols) == 4:
            hasil.append({
                'satuan'    : cols[0],
                'antam'     : cols[1],
                'pegadaian' : cols[2],
                'pluang'    : cols[3],
            })
    
    return hasil

# ── Test satu tanggal dulu ──
data = scrape_harga_emas(2026, 5, 1)
if data:
    df = pd.DataFrame(data)
    display(df)

  Final URL: https://harga-emas.org/history-harga/2026/Mei/01 | Status: 307


In [97]:
BULAN_ID = {
    1: 'Januari', 2: 'Februari', 3: 'Maret', 4: 'April',
    5: 'Mei', 6: 'Juni', 7: 'Juli', 8: 'Agustus',
    9: 'September', 10: 'Oktober', 11: 'November', 12: 'Desember'
}

# Coba manual follow redirect — ambil Location header dari 307
tahun, bulan, tanggal = 2026, 5, 2
nama_bulan = BULAN_ID[bulan]
url = f"https://harga-emas.org/history-harga/{tahun}/{nama_bulan}/{tanggal:02d}"

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

# Step 1: jangan follow redirect dulu, lihat apa yang dikirim server
resp = requests.get(url, headers=headers, timeout=10, allow_redirects=False)
print(f"Status: {resp.status_code}")
print(f"Location header: {resp.headers.get('Location', 'tidak ada')}")
print(f"Semua headers: {dict(resp.headers)}")

Status: 307
Location header: tidak ada
Semua headers: {'Date': 'Sun, 03 May 2026 15:14:39 GMT', 'Content-Type': 'text/html; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'X-Content-Type-Options': 'nosniff', 'Referrer-Policy': 'strict-origin-when-cross-origin', 'Content-Security-Policy': "default-src 'self'; script-src 'self'  'unsafe-inline' www.google-analytics.com https://www.googletagmanager.com https://d2r1yp2w7bby2u.cloudfront.net https://s.go-mpulse.net https://googleads.g.doubleclick.net; style-src 'self' 'unsafe-inline'; img-src 'self' data: https:; connect-src 'self' https://stats.g.doubleclick.net www.google-analytics.com https://www.googletagmanager.com https://d2r1yp2w7bby2u.cloudfront.net https://analytics.google.com https://www.google.com https://google.com https://c.go-mpulse.net https://*.akstat.io https://*.akamaihd.net https://trial-eum-clienttons-s.akamaihd.net https://financial-content-api.pluang.com http://harga-emas-backend.pluang-pro

In [98]:
BULAN_ID = {
    1: 'Januari', 2: 'Februari', 3: 'Maret', 4: 'April',
    5: 'Mei', 6: 'Juni', 7: 'Juli', 8: 'Agustus',
    9: 'September', 10: 'Oktober', 11: 'November', 12: 'Desember'
}

headers = {
    'User-Agent'      : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept'          : 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language' : 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
    'Accept-Encoding' : 'gzip, deflate, br',
    'Connection'      : 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest'  : 'document',
    'Sec-Fetch-Mode'  : 'navigate',
    'Sec-Fetch-Site'  : 'none',
    'Sec-Fetch-User'  : '?1',
}

tahun, bulan, tanggal = 2026, 5, 2
nama_bulan = BULAN_ID[bulan]
url = f"https://harga-emas.org/history-harga/{tahun}/{nama_bulan}/{tanggal:02d}"

# Pakai session agar cookies tersimpan antar request
session = requests.Session()
resp = session.get(url, headers=headers, timeout=10, allow_redirects=True)

print(f"Status: {resp.status_code}")
print(f"Final URL: {resp.url}")

soup = BeautifulSoup(resp.text, 'html.parser')
tables = soup.find_all('table')
print(f"Jumlah tabel: {len(tables)}")

if len(tables) >= 2:
    rows = tables[1].find_all('tr')
    for row in rows:
        cols = [td.get_text(strip=True) for td in row.find_all('td')]
        if cols:
            print(cols)

Status: 200
Final URL: https://harga-emas.org/history-harga/2026/Mei/02
Jumlah tabel: 2
['1000', 'Rp2.736.600.000', 'Rp2.702.112.000', 'Rp2.644.818.000']
['500', 'Rp1.368.320.000', 'Rp1.351.056.000', 'Rp1.322.409.000']
['250', 'Rp684.265.000', 'Rp675.529.000', 'Rp661.204.500']
['100', 'Rp273.812.000', 'Rp270.877.000', 'Rp264.481.800']
['50', 'Rp136.945.000', 'Rp135.505.000', 'Rp132.240.900']
['25', 'Rp68.512.000', 'Rp67.806.000', 'Rp66.120.450']
['10', 'Rp27.455.000', 'Rp27.270.000', 'Rp26.448.180']
['5', 'Rp13.755.000', 'Rp13.671.000', 'Rp13.224.090']
['2', 'Rp5.532.000', 'Rp5.509.000', 'Rp5.289.636']
['1', 'Rp2.796.000', 'Rp2.788.000', 'Rp2.644.818']
['0.5', 'Rp1.448.000', 'Rp1.462.000', 'Rp1.322.409']


### Bisa

In [118]:
BULAN_ID = {
    1: 'Januari', 2: 'Febuari', 3: 'Maret', 4: 'April',
    5: 'Mei', 6: 'Juni', 7: 'Juli', 8: 'Agustus',
    9: 'September', 10: 'Oktober', 11: 'November', 12: 'Desember'
}

def buat_session():
    session = requests.Session()
    retry = Retry(
        total=5,                        # max 3 kali retry
        backoff_factor=4,               # jeda: 2s, 4s, 8s
        status_forcelist=[429, 500, 502, 503, 504]  # retry kalau status ini
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('https://', adapter)
    session.headers.update({
        'User-Agent'               : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept'                   : 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language'          : 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept-Encoding'          : 'gzip, deflate, br',
        'Connection'               : 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Dest'           : 'document',
        'Sec-Fetch-Mode'           : 'navigate',
        'Sec-Fetch-Site'           : 'none',
        'Sec-Fetch-User'           : '?1',
    })
    return session

SESSION = buat_session()

def parse_harga(text):
    """'Rp2.796.000' → 2796000, kalau gagal return 0"""
    try:
        return int(text.replace('Rp', '').replace('.', '').replace(',', '').strip())
    except:
        return 0

def scrape_harga_emas(tahun, bulan, tanggal, max_retry=5):
    nama_bulan = BULAN_ID[bulan]
    url = f"https://harga-emas.org/history-harga/{tahun}/{nama_bulan}/{tanggal:02d}"

    for attempt in range(1, max_retry + 1):
        try:
            resp = SESSION.get(url, timeout=10, allow_redirects=True)

            if resp.status_code not in [200, 307]:
                print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] HTTP {resp.status_code} (attempt {attempt}/{max_retry})")
                if attempt < max_retry:
                    time.sleep(2 ** attempt)  # backoff: 2s, 4s, 8s
                    continue
                return None

            soup = BeautifulSoup(resp.text, 'html.parser')
            tables = soup.find_all('table')

            harga_table = None
            for table in tables:
                headers_row = [th.get_text(strip=True) for th in table.find_all('th')]
                if any('Antam' in h or 'Pegadaian' in h or 'Pluang' in h for h in headers_row):
                    harga_table = table
                    break

            if harga_table is None:
                # Tidak di-retry — kemungkinan memang tidak ada data (libur/weekend)
                return None

            all_headers = [th.get_text(strip=True) for th in harga_table.find_all('th')]

            def cari_index(keyword):
                for i, h in enumerate(all_headers):
                    if keyword.lower() in h.lower():
                        return i
                return None

            idx_antam     = cari_index('Antam')
            idx_pegadaian = cari_index('Pegadaian')
            idx_pluang    = cari_index('Pluang')

            hasil = []
            for row in harga_table.find_all('tr'):
                cols = row.find_all('td')
                if len(cols) < 2:
                    continue
                satuan = cols[0].get_text(strip=True)
                if not any(c.isdigit() for c in satuan):
                    continue

                def get_col(idx):
                    if idx is not None and idx < len(cols):
                        return parse_harga(cols[idx].get_text(strip=True))
                    return 0

                hasil.append({
                    'tanggal'   : f"{tahun}-{bulan:02d}-{tanggal:02d}",
                    'satuan_gr' : satuan,
                    'antam'     : get_col(idx_antam),
                    'pegadaian' : get_col(idx_pegadaian),
                    'pluang'    : get_col(idx_pluang),
                })

            return hasil if hasil else None

        except requests.exceptions.Timeout:
            print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] Timeout (attempt {attempt}/{max_retry})")
            if attempt < max_retry:
                time.sleep(2 ** attempt)
        except requests.exceptions.ConnectionError:
            print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] Connection error (attempt {attempt}/{max_retry})")
            if attempt < max_retry:
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] Error: {e}")
            return None  # error tak terduga, langsung skip

    return None


def scrape_range(start_date, end_date, delay=1.0, output_dir='data_emas'):
    os.makedirs(output_dir, exist_ok=True)
    
    start   = datetime.strptime(start_date, '%Y-%m-%d')
    end     = datetime.strptime(end_date,   '%Y-%m-%d')
    current = start

    while current <= end:
        tahun = current.year
        bulan = current.month
        nama_bulan = BULAN_ID[bulan]
        output_file = f"{output_dir}/{tahun}_{bulan:02d}.parquet"

        # Skip kalau bulan ini sudah pernah di-scrape
        if os.path.exists(output_file):
            print(f"[SKIP] {tahun}/{nama_bulan} sudah ada, lewat...")
            # Maju ke bulan berikutnya
            if bulan == 12:
                current = current.replace(year=tahun+1, month=1, day=1)
            else:
                current = current.replace(month=bulan+1, day=1)
            continue

        # Scrape semua hari dalam bulan ini
        bulan_rows = []
        while current.month == bulan and current <= end:
            label = current.strftime('%Y-%m-%d')
            print(f"  Fetching {label}...", end=' ')

            hasil = scrape_harga_emas(current.year, current.month, current.day)
            if hasil:
                bulan_rows.extend(hasil)
                print(f"✓ {len(hasil)} baris")
            else:
                print("(skip)")

            current += timedelta(days=1)
            time.sleep(delay)

        # Simpan checkpoint per bulan
        if bulan_rows:
            df_bulan = pl.DataFrame(bulan_rows)
            df_bulan = df_bulan.with_columns(
                pl.col('tanggal').str.to_date('%Y-%m-%d')
            )
            df_bulan.write_parquet(output_file)
            print(f"  ✓ Saved {output_file} ({len(df_bulan):,} baris)\n")

    # Gabungkan semua file parquet jadi satu
    print("Menggabungkan semua file...")
    all_files = sorted([f"{output_dir}/{f}" for f in os.listdir(output_dir) if f.endswith('.parquet')])
    df_final = pl.concat([pl.read_parquet(f) for f in all_files])
    df_final = df_final.sort(['tanggal', 'satuan_gr'])
    df_final.write_parquet(f"{output_dir}/harga_emas_FINAL.parquet")
    
    # Menggunakan Polars syntax
    print(f"\n✓ Selesai! Total {len(df_final):,} baris, {df_final['tanggal'].n_unique():,} hari")
    print(f"✓ Tersimpan di {output_dir}/harga_emas_FINAL.parquet")
    return df_final


# ── Test satu tanggal dulu ──
tanggal_awal = '2018-02-01'
tanggal_akhir = '2026-05-03'
delay = 1.0
output_file = 'data_emas_hargaemas-org'
hasil = scrape_range(tanggal_awal, tanggal_akhir, delay, output_file)
# df_test = pd.DataFrame(hasil)
# df_test.to_csv('emas.csv', index=False)

  Fetching 2018-02-01... ✓ 11 baris
  Fetching 2018-02-02... ✓ 11 baris
  Fetching 2018-02-03... ✓ 11 baris
  Fetching 2018-02-04... (skip)
  Fetching 2018-02-05... (skip)
  Fetching 2018-02-06... ✓ 11 baris
  Fetching 2018-02-07... ✓ 11 baris
  Fetching 2018-02-08... ✓ 11 baris
  Fetching 2018-02-09... ✓ 11 baris
  Fetching 2018-02-10... ✓ 11 baris
  Fetching 2018-02-11... (skip)
  Fetching 2018-02-12... (skip)
  Fetching 2018-02-13... ✓ 11 baris
  Fetching 2018-02-14... ✓ 11 baris
  Fetching 2018-02-15... ✓ 11 baris
  Fetching 2018-02-16... ✓ 11 baris
  Fetching 2018-02-17... (skip)
  Fetching 2018-02-18... (skip)
  Fetching 2018-02-19... (skip)
  Fetching 2018-02-20... ✓ 11 baris
  Fetching 2018-02-21... ✓ 11 baris
  Fetching 2018-02-22... ✓ 11 baris
  Fetching 2018-02-23... ✓ 11 baris
  Fetching 2018-02-24... ✓ 11 baris
  Fetching 2018-02-25... (skip)
  Fetching 2018-02-26... (skip)
  Fetching 2018-02-27... ✓ 11 baris
  Fetching 2018-02-28... ✓ 11 baris
  ✓ Saved data_emas_hargaema

In [ ]:
data_emas = pd.read_csv('emas.csv')
display(data_emas)

,tanggal,satuan_gr,antam,pegadaian,pluang
0,2026-04-01,0.5,1501000,1526000,1356859
1,2026-04-01,1.0,2902000,2908000,2713718
2,2026-04-01,10.0,28515000,28447000,27137180
3,2026-04-01,100.0,284412000,282581000,271371800
4,2026-04-01,1000.0,2842600000,2818862000,2713718000
...,...,...,...,...,...
325,2026-05-03,25.0,68512000,67806000,0
326,2026-05-03,250.0,684265000,675529000,0
327,2026-05-03,5.0,14728000,13671000,0
328,2026-05-03,50.0,136945000,135505000,0


In [112]:
# Debug: lihat redirect chain-nya
resp = SESSION.get(url, timeout=10, allow_redirects=True)
print(f"Final URL: {resp.url}")
print(f"Redirect history:")
for r in resp.history:
    print(f"  {r.status_code} → {r.headers.get('Location', '?')}")
print(f"Final status: {resp.status_code}")
print(f"Final content length: {len(resp.text)}")

Final URL: https://harga-emas.org/history-harga/2026/Mei/02
Redirect history:
Final status: 307
Final content length: 73299


In [113]:
SESSION = buat_session()  # fresh session
resp = SESSION.get("https://harga-emas.org", timeout=10)  # hit homepage dulu
print(f"Homepage: {resp.status_code}")
print(f"Cookies: {dict(SESSION.cookies)}")

# Baru hit URL target
resp2 = SESSION.get(url, timeout=10, allow_redirects=True)
print(f"Target: {resp2.status_code} | {resp2.url}")

Homepage: 200
Cookies: {}
Target: 307 | https://harga-emas.org/history-harga/2026/Mei/02


In [117]:
# Test langsung untuk tanggal bermasalah
resp = SESSION.get(
    "https://harga-emas.org/history-harga/2018/Febuari/01",
    timeout=10,
    allow_redirects=True
)
print(f"Status: {resp.status_code}")

soup = BeautifulSoup(resp.text, 'html.parser')
tables = soup.find_all('table')
print(f"Jumlah tabel: {len(tables)}")

for i, table in enumerate(tables):
    headers_row = [th.get_text(strip=True) for th in table.find_all('th')]
    print(f"Tabel {i} headers: {headers_row}")

Status: 200
Jumlah tabel: 2
Tabel 0 headers: ['Ming', 'Sen', 'Sel', 'Rab', 'Kam', 'Jum', 'Sab']
Tabel 1 headers: ['Satuan', 'Harga Emas Pluang', 'Gram', 'per Batangan (Rp)']
